In [1]:
# =========================
# Notebook 01: Airbnb cleaned + geohash
# =========================

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [2]:
# ---- Paths ----
HDFS_BASE = "hdfs:///user/vagrant"  # <- jeśli trzeba, zmień na inny prefix (np. hdfs:///)
AIRBNB_INPUT = f"{HDFS_BASE}/bigdata/processed/airbnb_listings/*.csv"
AIRBNB_OUTPUT = f"{HDFS_BASE}/bigdata/processed/airbnb_cleaned"

print("Input :", AIRBNB_INPUT)
print("Output:", AIRBNB_OUTPUT)

Input : hdfs:///user/vagrant/bigdata/processed/airbnb_listings/*.csv
Output: hdfs:///user/vagrant/bigdata/processed/airbnb_cleaned


In [3]:
# ---- Read Parquet ----

airbnb_raw = spark.read.parquet(AIRBNB_INPUT)

print("Columns:", len(airbnb_raw.columns))
airbnb_raw.printSchema()
airbnb_raw.show(3, truncate=80)

Columns: 65
root
 |-- id: string (nullable = true)
 |-- scrape_id: string (nullable = true)
 |-- last_scraped_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since_id: integer (nullable = true)
 |-- host_location_country: string (nullable = true)
 |-- host_location_city: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: integer (nullable = true)
 |-- host_response_rate: float (nullable = true)
 |-- host_acceptance_rate: float (nullable = true)
 |-- host_is_superhost: boolean (nullable = true)
 |-- host_listings_count: integer (nullable = true)
 |-- host_total_listings_count: integer (nullable = true)
 |-- host_has_profile_pic: boolean (nullable = true)
 |-- host_identity_verified: boolean (nullable = true)
 |-- neighbourhood_cleansed: string (n

26/01/13 02:02:17 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+------+---------+---------------+---------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+-------+---------+-------------+---------------------+------------------+--------------------------------------------------------------------------------+------------------+------------------+--------------------+-----------------+-------------------+-------------------------+--------------------+----------------------+----------------------+--------+---------+------------------+---------+------------+---------+--------+----+-----+--------------+--------------+---------------+---------------+---------------+----------------+-----------------+---------------------+----------------------+-----------------------+---------------+--------------+--------------------+----------------------+-------------------------+---------------------+------------------------

In [4]:
from pyspark.sql import functions as F

airbnb_raw.withColumn("_file", F.input_file_name()) \
    .groupBy("_file").count() \
    .show(truncate=False)

[Stage 11:============================================>           (60 + 3) / 75]

+-------------------------------------------------------------------------------------------------+-----+
|_file                                                                                            |count|
+-------------------------------------------------------------------------------------------------+-----+
|hdfs://node1/user/vagrant/bigdata/processed/airbnb_listings/listings_new-york-city_2025-10-01.csv|200  |
|hdfs://node1/user/vagrant/bigdata/processed/airbnb_listings/listings_london_2025-06-10.csv       |200  |
+-------------------------------------------------------------------------------------------------+-----+



In [5]:
# ---- Basic profiling / sanity checks ----
print("Total rows:", airbnb_raw.count())

airbnb_raw.select(
    F.count(F.lit(1)).alias("rows"),
    F.sum(F.col("latitude").isNull().cast("int")).alias("lat_nulls"),
    F.sum(F.col("longitude").isNull().cast("int")).alias("lon_nulls"),
    F.sum(F.col("id").isNull().cast("int")).alias("id_nulls"),
).show()

Total rows: 400
+----+---------+---------+--------+
|rows|lat_nulls|lon_nulls|id_nulls|
+----+---------+---------+--------+
| 400|        0|        0|       0|
+----+---------+---------+--------+



In [6]:
# ---- Helper: parse city from filename + map to city_code ----
# Example filename: listings_new-york-city_2025-10-01.csv

airbnb_with_file = airbnb_raw.withColumn("_input_file", F.input_file_name())

# Extract "new-york-city" part
airbnb_with_city = airbnb_with_file.withColumn(
    "city_slug",
    F.regexp_extract(F.col("_input_file"), r"listings_([^_]+)_\d{4}-\d{2}-\d{2}\.csv", 1)
)

# Map slug -> city_code
airbnb_with_city = airbnb_with_city.withColumn(
    "city_code",
    F.when(F.col("city_slug") == F.lit("new-york-city"), F.lit("NYC"))
     .when(F.col("city_slug") == F.lit("london"), F.lit("LON"))
     .when(F.col("city_slug") == F.lit("bristol"), F.lit("BRS"))
     .otherwise(F.lit("UNK"))
)

airbnb_with_city.select("city_slug", "city_code").distinct().show(truncate=False)

+-------------+---------+
|city_slug    |city_code|
+-------------+---------+
|new-york-city|NYC      |
|london       |LON      |
+-------------+---------+



In [7]:
# ---- Clean core fields: latitude/longitude + id ----
# 1) drop null lat/lon/id
# 2) cast lat/lon to double (czasem są float/string)
# 3) validate ranges

airbnb_geo = (
    airbnb_with_city
    .withColumn("latitude_d", F.col("latitude").cast("double"))
    .withColumn("longitude_d", F.col("longitude").cast("double"))
    .filter(F.col("id").isNotNull())
    .filter(F.col("latitude_d").isNotNull() & F.col("longitude_d").isNotNull())
    .filter((F.col("latitude_d") >= -90) & (F.col("latitude_d") <= 90))
    .filter((F.col("longitude_d") >= -180) & (F.col("longitude_d") <= 180))
)

print("Rows after geo/id filtering:", airbnb_geo.count())

Rows after geo/id filtering: 400


In [8]:
# ---- Price parsing (robust) ----

def parse_price(col):
    return F.regexp_replace(F.regexp_replace(col.cast("string"), r"[\$,]", ""), r"\s+", "")

airbnb_price = airbnb_geo.withColumn("price_num", parse_price(F.col("price")).cast("double"))

airbnb_price.select("price", "price_num").show(5, truncate=False)

+-----+---------+
|price|price_num|
+-----+---------+
|297.0|297.0    |
|98.0 |98.0     |
|148.0|148.0    |
|144.0|144.0    |
|157.0|157.0    |
+-----+---------+
only showing top 5 rows



In [9]:
# ---- Optional: dedup by id ----

airbnb_dedup = airbnb_price.dropDuplicates(["id"])
print("Rows after dedup:", airbnb_dedup.count())

[Stage 30:==================================================>   (186 + 2) / 200]

Rows after dedup: 400


In [10]:
# ---- Geohash5 UDF (pure python, no external libs) ----

_base32 = "0123456789bcdefghjkmnpqrstuvwxyz"

def geohash_encode(lat, lon, precision=5):
    if lat is None or lon is None:
        return None
    try:
        lat = float(lat); lon = float(lon)
    except Exception:
        return None

    lat_interval = [-90.0, 90.0]
    lon_interval = [-180.0, 180.0]
    geohash = []
    is_even = True
    bit = 0
    ch = 0
    bits = [16, 8, 4, 2, 1]

    while len(geohash) < precision:
        if is_even:
            mid = (lon_interval[0] + lon_interval[1]) / 2
            if lon > mid:
                ch |= bits[bit]
                lon_interval[0] = mid
            else:
                lon_interval[1] = mid
        else:
            mid = (lat_interval[0] + lat_interval[1]) / 2
            if lat > mid:
                ch |= bits[bit]
                lat_interval[0] = mid
            else:
                lat_interval[1] = mid

        is_even = not is_even
        if bit < 4:
            bit += 1
        else:
            geohash.append(_base32[ch])
            bit = 0
            ch = 0

    return "".join(geohash)

geohash_udf = F.udf(lambda lat, lon: geohash_encode(lat, lon, precision=5), T.StringType())

airbnb_geohash = airbnb_dedup.withColumn("geohash5", geohash_udf(F.col("latitude_d"), F.col("longitude_d")))

airbnb_geohash.select("id", "city_code", "latitude_d", "longitude_d", "geohash5").show(5, truncate=False)

+------+---------+-----------------+-------------------+--------+
|id    |city_code|latitude_d       |longitude_d        |geohash5|
+------+---------+-----------------+-------------------+--------+
|16974 |NYC      |40.80189895629883|-73.9376220703125  |dr72j   |
|429444|LON      |51.4788818359375 |-0.1071000024676323|gcpuv   |
|89427 |NYC      |40.68513107299805|-73.96647644042969 |dr5rk   |
|21456 |NYC      |40.79935836791992|-73.9615478515625  |dr72h   |
|24285 |NYC      |40.67115020751953|-73.98026275634766 |dr5rk   |
+------+---------+-----------------+-------------------+--------+
only showing top 5 rows



In [11]:
# ---- Select final columns (core analytical set) ----

keep_cols = [
    "id",
    "city_code",
    "neighbourhood_cleansed",
    "latitude_d",
    "longitude_d",
    "geohash5",
    "price_num",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "number_of_reviews",
    "review_scores_rating",
    "availability_365",
    "host_is_superhost",
    "host_response_rate",
    "host_acceptance_rate",
]

existing = [c for c in keep_cols if c in airbnb_geohash.columns]
airbnb_cleaned = airbnb_geohash.select(*existing)

airbnb_cleaned.printSchema()
airbnb_cleaned.show(5, truncate=80)

root
 |-- id: string (nullable = true)
 |-- city_code: string (nullable = false)
 |-- neighbourhood_cleansed: string (nullable = true)
 |-- latitude_d: double (nullable = true)
 |-- longitude_d: double (nullable = true)
 |-- geohash5: string (nullable = true)
 |-- price_num: double (nullable = true)
 |-- room_type: integer (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- bedrooms: float (nullable = true)
 |-- beds: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- review_scores_rating: float (nullable = true)
 |-- availability_365: integer (nullable = true)
 |-- host_is_superhost: boolean (nullable = true)
 |-- host_response_rate: float (nullable = true)
 |-- host_acceptance_rate: float (nullable = true)

+------+---------+----------------------+-----------------+-------------------+--------+---------+---------+---------------------------+------------+--------+----+-----------------+---------

In [12]:
# ---- Write Parquet to HDFS ----

(airbnb_cleaned
 .write
 .mode("overwrite")
 .parquet(AIRBNB_OUTPUT)
)

print("Saved to:", AIRBNB_OUTPUT)

26/01/13 02:03:24 WARN hdfs.DFSClient: Caught exception          (12 + 2) / 200]
java.lang.InterruptedException
	at java.lang.Object.wait(Native Method)
	at java.lang.Thread.join(Thread.java:1252)
	at java.lang.Thread.join(Thread.java:1326)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.closeResponder(DFSOutputStream.java:716)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.endBlock(DFSOutputStream.java:476)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.run(DFSOutputStream.java:652)
26/01/13 02:03:31 WARN hdfs.DFSClient: Caught exception          (47 + 2) / 200]
java.lang.InterruptedException
	at java.lang.Object.wait(Native Method)
	at java.lang.Thread.join(Thread.java:1252)
	at java.lang.Thread.join(Thread.java:1326)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.closeResponder(DFSOutputStream.java:716)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.endBlock(DFSOutputStream.java:476)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.

Saved to: hdfs:///user/vagrant/bigdata/processed/airbnb_cleaned


In [13]:
# ---- Verify write ----
airbnb_check = spark.read.parquet(AIRBNB_OUTPUT)
print("Rows written:", airbnb_check.count())
airbnb_check.select("city_code").groupBy("city_code").count().show()
airbnb_check.show(3, truncate=80)

Rows written: 400


+---------+-----+
|city_code|count|
+---------+-----+
|      LON|  200|
|      NYC|  200|
+---------+-----+

+------+---------+----------------------+-----------------+--------------------+--------+---------+---------+---------------------------+------------+--------+----+-----------------+--------------------+----------------+-----------------+------------------+--------------------+
|    id|city_code|neighbourhood_cleansed|       latitude_d|         longitude_d|geohash5|price_num|room_type|              property_type|accommodates|bedrooms|beds|number_of_reviews|review_scores_rating|availability_365|host_is_superhost|host_response_rate|host_acceptance_rate|
+------+---------+----------------------+-----------------+--------------------+--------+---------+---------+---------------------------+------------+--------+----+-----------------+--------------------+----------------+-----------------+------------------+--------------------+
| 13913|      LON|             Islington|51.5686111450